# 06b-l — revisione atomica del modello d'errore di voltaggio

Ultima disambiguazione prima della revisione architetturale: gate causali hard/soft tra persistenza e update dinamico, confrontati con due oracoli teacher non selezionabili.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Sorgenti immutabili

Servono dataset composito, catena 05t–06b-j e risultato esatto 06b-k. Ogni artefatto viene identificato dal digest del proprio indice.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source,materialize_nested_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model.causal_voltage_state_coupling_forensic import EXPECTED_06B_INDEX_SHA256
from src.hayflow_model.causal_voltage_bridge_representation_forensic import EXPECTED_06BB_INDEX_SHA256
from src.hayflow_model.nested_coupling_optimization_scaling_forensic import EXPECTED_06BC_INDEX_SHA256
from src.hayflow_model.recursive_voltage_state_contract_forensic import EXPECTED_06BD_INDEX_SHA256
from src.hayflow_model.recursive_joint_repair_matrix import EXPECTED_06BE_INDEX_SHA256
from src.hayflow_model.state_scheduled_sampling_confirmation import EXPECTED_06BF_INDEX_SHA256
from src.hayflow_model.frozen_voltage_generalization_forensic import EXPECTED_06BG_INDEX_SHA256
from src.hayflow_model.voltage_objective_recalibration_playground import EXPECTED_06BH_INDEX_SHA256
from src.hayflow_model.analytic_causal_gain_identifiability import EXPECTED_06BI_INDEX_SHA256
from src.hayflow_model.temporal_voltage_correction_state import EXPECTED_06BJ_INDEX_SHA256
from src.hayflow_model.voltage_error_model_revision import EXPECTED_06BK_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
ARTIFACT_NAMES={'05t':'hayflow_consolidated_autoregressive_go_no_go','06b':'hayflow_optimized_explicit_state_updater_canary','06b-b':'hayflow_causal_voltage_state_coupling_forensic','06b-c':'hayflow_causal_voltage_bridge_representation_forensic','06b-d':'hayflow_nested_coupling_optimization_scaling_forensic','06b-e':'hayflow_recursive_voltage_state_contract_forensic','06b-f':'hayflow_recursive_joint_repair_matrix','06b-g':'hayflow_state_scheduled_sampling_confirmation','06b-h':'hayflow_frozen_voltage_generalization_forensic','06b-i':'hayflow_voltage_objective_recalibration_playground','06b-j':'hayflow_analytic_causal_gain_identifiability','06b-k':'hayflow_temporal_voltage_correction_state'}
missing_artifacts=[]
def indexed(name,expected,env):
 override=os.environ.get(env);source=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override) if override else None)
 if source is None:source=materialize_nested_indexed_artifact_source(INPUT_ROOT,expected,Path('/kaggle/working/.06bl_nested_inputs'))
 if source is None:missing_artifacts.append(f'{name}: {ARTIFACT_NAMES[name]} ({env})')
 return source
ARTIFACT_05T_SOURCE=indexed('05t',EXPECTED_05T_INDEX_SHA256,'HAYFLOW_05T_ARTIFACT');ARTIFACT_06B_SOURCE=indexed('06b',EXPECTED_06B_INDEX_SHA256,'HAYFLOW_06B_ARTIFACT');ARTIFACT_06BB_SOURCE=indexed('06b-b',EXPECTED_06BB_INDEX_SHA256,'HAYFLOW_06BB_ARTIFACT');ARTIFACT_06BC_SOURCE=indexed('06b-c',EXPECTED_06BC_INDEX_SHA256,'HAYFLOW_06BC_ARTIFACT');ARTIFACT_06BD_SOURCE=indexed('06b-d',EXPECTED_06BD_INDEX_SHA256,'HAYFLOW_06BD_ARTIFACT')
override_06be=os.environ.get('HAYFLOW_06BE_ARTIFACT');ARTIFACT_06BE_SOURCE=None
for expected in EXPECTED_06BE_INDEX_SHA256:
 candidate=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override_06be) if override_06be else None)
 if candidate is None:candidate=materialize_nested_indexed_artifact_source(INPUT_ROOT,expected,Path('/kaggle/working/.06bl_nested_inputs'))
 if candidate is not None:ARTIFACT_06BE_SOURCE=candidate;break
if ARTIFACT_06BE_SOURCE is None:missing_artifacts.append(f"06b-e: {ARTIFACT_NAMES['06b-e']} (HAYFLOW_06BE_ARTIFACT)")
ARTIFACT_06BF_SOURCE=indexed('06b-f',EXPECTED_06BF_INDEX_SHA256,'HAYFLOW_06BF_ARTIFACT');ARTIFACT_06BG_SOURCE=indexed('06b-g',EXPECTED_06BG_INDEX_SHA256,'HAYFLOW_06BG_ARTIFACT');ARTIFACT_06BH_SOURCE=indexed('06b-h',EXPECTED_06BH_INDEX_SHA256,'HAYFLOW_06BH_ARTIFACT');ARTIFACT_06BI_SOURCE=indexed('06b-i',EXPECTED_06BI_INDEX_SHA256,'HAYFLOW_06BI_ARTIFACT');ARTIFACT_06BJ_SOURCE=indexed('06b-j',EXPECTED_06BJ_INDEX_SHA256,'HAYFLOW_06BJ_ARTIFACT');ARTIFACT_06BK_SOURCE=indexed('06b-k',EXPECTED_06BK_INDEX_SHA256,'HAYFLOW_06BK_ARTIFACT')
assert not missing_artifacts,'Artefatti storici esatti non trovati: '+', '.join(missing_artifacts)+'. Aggiungili agli Input Kaggle oppure imposta le variabili HAYFLOW_* indicate.'
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06bl_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.'
print({'05t':str(ARTIFACT_05T_SOURCE),'06b':str(ARTIFACT_06B_SOURCE),'06b-b':str(ARTIFACT_06BB_SOURCE),'06b-c':str(ARTIFACT_06BC_SOURCE),'06b-d':str(ARTIFACT_06BD_SOURCE),'06b-e':str(ARTIFACT_06BE_SOURCE),'06b-f':str(ARTIFACT_06BF_SOURCE),'06b-g':str(ARTIFACT_06BG_SOURCE),'06b-h':str(ARTIFACT_06BH_SOURCE),'06b-i':str(ARTIFACT_06BI_SOURCE),'06b-j':str(ARTIFACT_06BJ_SOURCE),'06b-k':str(ARTIFACT_06BK_SOURCE),'base':str(BASE_SOURCE)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06b-l][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880;print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Gate di miscela e decisione architetturale

I gate causali sono stimati sul fit autoregressivo e selezionati sulla calibration train riusata. Qualunque esito finale determina direttamente una revisione dell'architettura.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import VoltageErrorModelRevision,VoltageErrorModelRevisionConfig
cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_recursive_joint_repair_matrix.yml').read_text())['recursive_joint_repair_matrix']
for filename,key in [('hayflow_state_scheduled_sampling_confirmation.yml','state_scheduled_sampling_confirmation'),('hayflow_frozen_voltage_generalization_forensic.yml','frozen_voltage_generalization_forensic'),('hayflow_voltage_objective_recalibration_playground.yml','voltage_objective_recalibration_playground'),('hayflow_analytic_causal_gain_identifiability.yml','analytic_causal_gain_identifiability'),('hayflow_temporal_voltage_correction_state.yml','temporal_voltage_correction_state'),('hayflow_voltage_error_model_revision.yml','voltage_error_model_revision')]:cfg.update(yaml.safe_load((ELM_REPO/'configs/hayflow'/filename).read_text())[key])
config=VoltageErrorModelRevisionConfig.from_mapping(cfg);OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_voltage_error_model_revision');assert not OUTPUT_DIR.exists(),f'Output gia presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=VoltageErrorModelRevision(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,ARTIFACT_06B_SOURCE,ARTIFACT_06BB_SOURCE,ARTIFACT_06BC_SOURCE,ARTIFACT_06BD_SOURCE,ARTIFACT_06BE_SOURCE,ARTIFACT_06BF_SOURCE,ARTIFACT_06BG_SOURCE,ARTIFACT_06BH_SOURCE,ARTIFACT_06BI_SOURCE,ARTIFACT_06BJ_SOURCE,ARTIFACT_06BK_SOURCE,code_revision=REVISION);contract=session.prepare_voltage_error_model_revision()
display({'valid':contract['valid'],'06b-k':contract['source_06bk'],'experts':contract['experts'],'causal_gates':contract['causal_gate_schemes'],'primary':contract['primary_scheme'],'fallback':contract['fallback_scheme'],'oracles':contract['oracle_schemes'],'oracles_selectable':contract['oracles_eligible_for_selection'],'terminal':contract['terminal_diagnostic_before_architecture_revision'],'neural_training':contract['neural_training_performed']});assert contract['valid'] and contract['terminal_diagnostic_before_architecture_revision'] and not contract['oracles_eligible_for_selection']

In [ ]:
calibration_report=session.fit_and_calibrate_gate_models();display({'valid':calibration_report['valid'],'fit_role':calibration_report['fit_role'],'selection_role':calibration_report['selection_role'],'development_accessed':calibration_report['development_accessed'],'oracles_used':calibration_report['oracles_used_during_selection'],'selected':{seed:{scheme:row['selected'] for scheme,row in schemes.items()} for seed,schemes in calibration_report['per_seed'].items()}});assert calibration_report['valid'] and not calibration_report['development_accessed'] and not calibration_report['oracles_used_during_selection']

In [ ]:
evaluation_report=session.evaluate_voltage_error_models();final_report=session.finalize_voltage_error_model_revision(evaluation_report)
compact={scheme:{'V_vs_static':round(row['median_recursive_gain_over_static_fraction'],4),'V_vs_persistence':round(row['median_voltage_gain_vs_persistence_fraction'],4),'STATE_vs_persistence':round(row['median_STATE_gain_vs_persistence_fraction'],4),'quiescent_gain':round(row['activity_gain_vs_persistence']['quiescent_lt_1mV'],4),'quiescent_rmse_mv':round(row['activity_rmse_mv']['quiescent_lt_1mV'],4),'moderate_gain':round(row['activity_gain_vs_persistence']['moderate_1_to_5mV'],4),'active_gain':round(row['activity_gain_vs_persistence']['active_ge_5mV'],4),'soma_gain':round(row['region_gain_vs_persistence']['soma'],4),'passed':row['registered_gate_passed']} for scheme,row in final_report['summaries'].items()}
display(compact);display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'selected_causal_scheme':final_report['selected_causal_scheme'],'primary_passed':final_report['primary_passed'],'fallback_passed':final_report['fallback_passed'],'other_causal_passes':final_report['other_causal_passing_schemes'],'teacher_regime_oracle_passed':final_report['teacher_regime_oracle_passed'],'optimal_blend_oracle_passed':final_report['teacher_optimal_blend_oracle_passed'],'oracles_selectable':final_report['oracles_eligible_for_selection'],'next_step':final_report['next_step']});assert final_report['valid'] and final_report['terminal_diagnostic_before_architecture_revision'] and not final_report['oracles_eligible_for_selection']

## 3. Download stabile

La cella crea lo ZIP in `/kaggle/working`, lo converte in base64 e avvia il download tramite un Blob nel browser.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_voltage_error_model_revision','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})